# DEPURACIÓN Y PREPROCESAMIENTO DEL DATASET DE CONSUMO ENERGÉTICO

## Objetivo general
Este notebook constituye la fase fundamental de **limpieza y estructuración de datos** dentro del pipeline de ciencia de datos. Su propósito principal es tomar el conjunto de datos crudo (`dataset_consumo_energetico_CRUDO.csv`), estandarizar sus atributos, corregir errores de captura y depurar valores lógicamente inconsistentes, dejando la base de datos completamente normalizada y preparada para las fases de **imputación de valores nulos**, **análisis exploratorio (EDA)** y el posterior **desarrollo de modelos predictivos**.

---

## Flujo metodológico y procesos aplicados

1. **Eliminación de registros duplicados**
   - Detección y eliminación de filas completamente idénticas, conservando únicamente la primera aparición para evitar sesgos por sobre-representación de registros.
   - Eliminación de columnas residuales sin valor analítico (como índices heredados de importaciones previas, e.g., `Unnamed: 0`) y reinicio de la numeración del índice.

2. **Conversión y estandarización de tipos de datos:**
   Alineación de cada atributo al diccionario de datos del proyecto aprovechando los tipos anulables de **Pandas**, los cuales permiten conservar datos faltantes (`NaN`) sin alterar la naturaleza de las variables:
   - **Variables categóricas (texto):** Convertidas al tipo `string` para un procesamiento de cadenas uniforme.
   - **Variables numéricas (enteras y decimales):** Transformadas a `Int64` y `Float64` respectivamente, forzando como nulo cualquier error tipográfico no convertible (`errors="coerce"`).
   - **Variables booleanas:** Normalización de cadenas lógicas heterogéneas (`true`, `false`, `sí`, `no`, `1`, `0`) a una representación numérica binaria estricta de **1** y **0** en formato `Int64`, optimizando el dataset para el consumo en algoritmos de machine learning.

3. **Estandarización de variables categóricas:**
   - Depuración de errores tipográficos, diferencias entre mayúsculas y minúsculas, caracteres especiales y variaciones en la codificación.
   - Mapeo estructurado contra diccionarios de dominio controlado para unificar el dominio de categorías en atributos clave (ej. normalización de meses, tipos de inmueble, zonas geográficas y perfiles energéticos).

4. **Detección y estandarización de nulos fantasma:**
   - Identificación de convenciones erróneas utilizadas comúnmente para representar la ausencia de información (tales como `""`, `"?"`, `"N/A"`, `"NULL"`, `"-999"` o `"9999"`).
   - Reemplazo generalizado de estos marcadores por la representación estándar `np.nan` de NumPy, garantizando mediciones estadísticas confiables en los pasos siguientes.

5. **Depuración de outliers por dominio (validación física y lógica):**
   - Definición de fronteras teóricas admisibles `[mínimo, máximo]` de acuerdo a las leyes físicas del problema y el contexto tropical de la República Dominicana.
   - **Restricciones horarias:** Límites estrictos de 0 a 24 horas para variables de uso diario por equipo o ambiente, y máximos de 744 horas (31 días $\times$ 24h) y 31 días para variables de corte mensual.
   - **Restricciones de porcentaje y magnitudes:** Acotamiento de porcentajes de iluminación al rango $[0, 100]\%$ y eliminación de consumos, áreas o conteos con valores negativos.
   - *Nota metodológica:* En lugar de eliminar las filas completas con *outliers*, las celdas fuera de rango fueron neutralizadas e invalidadas a `np.nan`, preservando la información valiosa del resto de las columnas para su posterior tratamiento en la imputación.

6. **Depuración de estricta binaridad y eliminación de columnas completamente vacías y filas con variable predictiva vacía**
   - Auditoría final e individual sobre variables lógicas (`tiene_aire_acondicionado`, `tiene_lavadora`, etc.) para asegurar que su dominio se componga exclusivamente del conjunto de observaciones válidas $\{0, 1, \text{NaN}\}$.
   - Eliminación de registros con todos los valores vacíos y aquellos que tienen la variable "perfil_energetico" vacía.

---

## Resultado del pipeline
El proceso concluye preservando la integridad del volumen de registros útiles (99,024 filas $\times$ 43 columnas) y generando el archivo normalizado **`dataset_consumo_energetico_LIMPIO.csv`**, el cual representa la única fuente de verdad validada para alimentar las fases posteriores del proyecto.

## 1. Lectura de la base de datos

In [ ]:
import pandas as pd
import numpy as np
import os

RUTA_BD = ""
bd = pd.read_csv(RUTA_BD)

## 2. Eliminación de registros duplicados, columnas completamente vacías y filas con variable predictiva vacía.

Se identifican y eliminan las filas completamente duplicadas del conjunto de datos, comparando todas las columnas del DataFrame. Cuando existen múltiples registros idénticos, se conserva únicamente la primera aparición (`keep="first"`). Posteriormente, el índice del DataFrame se reinicia (`reset_index(drop=True)`) para mantener una numeración continua y evitar conservar los índices originales. Posteriormente se eliminan registros con todos los valores vacíos y aquellos que tienen la variable "perfil_energetico" vacía.

In [ ]:
bd.drop(axis=1, columns=["Unnamed: 0"], inplace=True)

In [ ]:
total_original = len(bd)

print("ELIMINACION DE VALORES DUPLICADOS")

bd.drop_duplicates(keep="first", inplace=True)
bd.reset_index(drop=True, inplace=True)

filas_borradas = total_original - len(bd)

print(f"Total de registros originales : {total_original:,}")
print(f"Filas 100% vacías eliminadas  : {filas_borradas:,}")
print(f"Total de registros limpios    : {len(bd):,}\n")

ELIMINACION DE VALORES DUPLICADOS
Total de registros originales : 99,024
Filas 100% vacías eliminadas  : 2,063
Total de registros limpios    : 96,961



## 3. Conversión de tipos de datos

Se realizó la conversión de los tipos de datos de cada variable para adaptarlos a las especificaciones establecidas en el diccionario de datos y asegurar su total compatibilidad con el estándar `np.nan`. El objetivo de este procedimiento fue garantizar que cada atributo del conjunto de datos posea un tipo de dato consistente con su naturaleza, respetando el diccionario y optimizando los requerimientos matriciales para el entrenamiento de modelos de aprendizaje automático.

El procedimiento se desarrolló en cuatro etapas:

1. **Conversión de variables categóricas (texto):** Las variables cualitativas fueron adaptadas al diccionario de datos y convertidas al tipo estándar `object` de Python/NumPy, permitiendo un tratamiento uniforme de la información textual y asegurando una compatibilidad directa con los algoritmos y matrices mediante el uso nativo de `np.nan`.

2. **Conversión de variables numéricas enteras:** Las variables que representan cantidades discretas fueron transformadas de acuerdo con el diccionario utilizando `pd.to_numeric()` con el parámetro `errors="coerce"`, de forma que cualquier valor no convertible fuese reemplazado por un valor nulo (`np.nan`). Posteriormente, dichas variables fueron almacenadas utilizando el tipo estándar `float64`, permitiendo conservar valores faltantes compatibles con NumPy sin alterar la naturaleza de la información.

3. **Conversión de variables numéricas decimales:** Las variables continuas fueron alineadas con las especificaciones del diccionario de datos y convertidas mediante `pd.to_numeric()`, almacenándose posteriormente bajo el tipo estándar `float64` para garantizar una representación adecuada de los valores reales y de los datos faltantes mediante `np.nan`.

4. **Conversión de variables binarias:** Las variables booleanas fueron transformadas de acuerdo con los criterios del diccionario de datos mediante un proceso de normalización que eliminó espacios innecesarios y unificó las distintas representaciones textuales de valores lógicos (por ejemplo, `"true"`, `"false"`, `"1"`, `"0"`, `"sí"` y `"no"`). Posteriormente, dichas variables fueron codificadas como **1** para representar el valor verdadero y **0** para representar el valor falso, almacenándose finalmente con el tipo `float64`. Esta representación numérica resulta idónea para cumplir con el diccionario y con el entrenamiento de modelos de aprendizaje automático basados en `np.nan`.

Como resultado del procedimiento, se generó un reporte con el tipo de dato final asignado a cada variable, permitiendo verificar que la estructura del conjunto de datos coincide rigurosamente con la definida en el diccionario de datos y con el estándar de compatibilidad `np.nan`. El conjunto de datos obtenido quedó preparado, optimizado y sin dependencias de `pd.NA`, listo para el desarrollo del Análisis Exploratorio de Datos (EDA) y la implementación de modelos predictivos.

In [ ]:
# ==========================================================
# CONVERSIÓN DE TIPOS DE DATOS SEGÚN EL DICCIONARIO
# (COMPATIBLE CON NUMPY Y MACHINE LEARNING / SIN pd.NA)
# DataFrame: bd
# ==========================================================

print("="*80)
print("CONVERSIÓN DE TIPOS DE DATOS")
print("="*80)

# ==========================================================
# Definición de variables por tipo
# ==========================================================

columnas_string = [
    "id_registro",
    "tipo_inmueble",
    "zona",
    "nivel_socioeconomico",
    "mes_referencia",
    "horario_pico_uso",
    "aislamiento_termico",
    "fuente_energia_secundaria",
    "perfil_energetico"
]

columnas_int = [
    "num_personas",
    "antiguedad_construccion_anios",
    "dias_facturacion",
    "cantidad_unidades_aa",
    "cantidad_tv_o_pantallas",
    "cantidad_computadoras",
    "cantidad_focos",
    "otros_equipos_pequenos",
    "cantidad_equipos_total",
    "dias_sin_electricidad_mes"
]

columnas_float = [
    "superficie_m2",
    "temperatura_promedio_c",
    "horas_uso_aa_dia",
    "pct_iluminacion_led",
    "horas_uso_iluminacion_dia",
    "horas_dia_cocina",
    "horas_dia_sala_estar",
    "horas_dia_dormitorios",
    "horas_dia_oficina_estudio",
    "horas_dia_lavanderia",
    "antiguedad_electrodomesticos_anios",
    "horas_uso_planta_o_inversor_mes",
    "generacion_solar_kwh_mensual",
    "consumo_kwh_mes_anterior",
    "variacion_pct_consumo_mensual",
    "consumo_kwh_mensual",
    "consumo_neto_facturado_kwh",
    "costo_estimado_usd",
    "consumo_kwh_por_m2",
    "consumo_kwh_por_persona"
]

columnas_bool = [
    "tiene_aire_acondicionado",
    "tiene_calentador_agua_electrico",
    "tiene_lavadora",
    "certificacion_energetica_previa"
]

# ==========================================================
# Conversión de variables tipo texto
# ==========================================================

for columna in columnas_string:

    if columna in bd.columns:

        bd[columna] = bd[columna].astype("object")

# ==========================================================
# Conversión de variables enteras
# ==========================================================

for columna in columnas_int:

    if columna in bd.columns:

        bd[columna] = (
            pd.to_numeric(
                bd[columna],
                errors="coerce"
            ).astype("float64")
        )

# ==========================================================
# Conversión de variables decimales
# ==========================================================

for columna in columnas_float:

    if columna in bd.columns:

        bd[columna] = (
            pd.to_numeric(
                bd[columna],
                errors="coerce"
            ).astype("float64")
        )

# ==========================================================
# Conversión de variables booleanas a 0 y 1
# ==========================================================

mapa_booleanos = {
    "true": 1,
    "false": 0,
    "1": 1,
    "0": 0,
    "si": 1,
    "sí": 1,
    "no": 0,
    "yes": 1,
    "y": 1,
    "n": 0,
    "true ": 1,
    "false ": 0,
    "nan": np.nan,
    "none": np.nan,
    "": np.nan
}

for columna in columnas_bool:

    if columna in bd.columns:

        # Si la columna ya es booleana
        if pd.api.types.is_bool_dtype(bd[columna]):

            bd[columna] = (
                bd[columna]
                .astype("float64")
            )

        else:

            bd[columna] = (
                bd[columna]
                .astype(str)
                .str.strip()
                .str.lower()
                .map(mapa_booleanos)
                .astype("float64")
            )

# ==========================================================
# Resumen
# ==========================================================

print("\nConversión finalizada correctamente.\n")

reporte = pd.DataFrame({
    "Variable": bd.columns,
    "Tipo final": [str(bd[col].dtype) for col in bd.columns]
})

display(reporte)

print("\nTipos de datos del DataFrame:\n")
print(bd.dtypes)

CONVERSIÓN DE TIPOS DE DATOS

Conversión finalizada correctamente.



,Variable,Tipo final
0,id_registro,object
1,tipo_inmueble,object
2,zona,object
3,nivel_socioeconomico,object
4,num_personas,float64
5,superficie_m2,float64
6,antiguedad_construccion_anios,float64
7,mes_referencia,object
8,dias_facturacion,float64
9,temperatura_promedio_c,float64



Tipos de datos del DataFrame:

id_registro                            object
tipo_inmueble                          object
zona                                   object
nivel_socioeconomico                   object
num_personas                          float64
superficie_m2                         float64
antiguedad_construccion_anios         float64
mes_referencia                         object
dias_facturacion                      float64
temperatura_promedio_c                float64
tiene_aire_acondicionado              float64
cantidad_unidades_aa                  float64
horas_uso_aa_dia                      float64
tiene_calentador_agua_electrico       float64
tiene_lavadora                        float64
cantidad_tv_o_pantallas               float64
cantidad_computadoras                 float64
pct_iluminacion_led                   float64
cantidad_focos                        float64
horas_uso_iluminacion_dia             float64
otros_equipos_pequenos                float64
ca

## 4. Estandarización de variables categóricas

## Objetivo

Este script tiene como propósito normalizar todas las variables categóricas del conjunto de datos utilizando un conjunto de categorías válidas previamente definidas. Durante el proceso se corrigen diferencias de escritura, formatos inconsistentes y errores de codificación, garantizando que cada variable posea un dominio uniforme antes de continuar con la imputación de valores faltantes y el análisis exploratorio de datos (EDA).

---

# Flujo de funcionamiento

## Paso 1. Definición de las categorías válidas

Se define un diccionario para cada variable categórica del dataset. Cada diccionario establece la correspondencia entre las distintas representaciones de una misma categoría y su valor estandarizado.

Por ejemplo:

- Diferencias entre mayúsculas y minúsculas.
- Abreviaturas.
- Meses escritos en distintos idiomas.
- Errores de codificación de caracteres.

Todas las variantes válidas apuntan a una única representación estándar.

---

## Paso 2. Recorrido de las variables categóricas

El script recorre automáticamente cada una de las variables categóricas incluidas en los diccionarios de estandarización.

Antes de realizar la sustitución de valores, cada columna es normalizada mediante:

- Conversión a tipo `string`.
- Eliminación de espacios en blanco al inicio y final.
- Conversión completa a letras minúsculas.

Esta normalización permite comparar correctamente las categorías independientemente de su formato original.

---

## Paso 3. Estandarización de categorías

Cada valor de la columna es comparado con el diccionario correspondiente.

Si la categoría existe dentro del dominio definido, se reemplaza automáticamente por su representación estándar.

Ejemplos:

- `APARTAMENTO` → `Apartamento`
- `apartamento` → `Apartamento`
- `Casa Unifamiliar` → `Casa Unifamiliar`
- `PequeÃ±o Establecimiento Comercial` → `Pequeño Establecimiento Comercial`
- `August` → `Agosto`
- `08` → `Agosto`

De esta manera todas las categorías quedan representadas mediante un único valor.

---

## Paso 4. Conversión de categorías inválidas

Si una categoría no pertenece al conjunto de valores definidos para la variable correspondiente, el script la considera inválida y la reemplaza automáticamente por `NaN`.

Este comportamiento elimina valores residuales, errores tipográficos y categorías no contempladas en el diccionario de datos.

---

## Paso 5. Actualización del dataset

Una vez finalizada la estandarización, cada columna original es reemplazada por su versión corregida, conservando el tipo de dato de texto (`string`) para facilitar el procesamiento posterior.

---

## Paso 6. Generación del reporte

Al finalizar el proceso se construye una tabla resumen que informa, para cada variable categórica:

- Nombre de la variable.
- Cantidad de valores corregidos.
- Cantidad de categorías convertidas a `NaN`.
- Número de categorías válidas presentes después de la estandarización.

Este reporte permite verificar el impacto del proceso de limpieza y confirmar que todas las variables poseen un dominio consistente.

---

# Resultado

Al concluir la ejecución del script:

- Todas las categorías equivalentes quedan representadas mediante un único valor estándar.
- Se corrigen automáticamente diferencias de escritura, uso de mayúsculas, abreviaturas y errores de codificación.
- Las categorías no reconocidas son transformadas en valores nulos (`NaN`).
- Se obtiene un conjunto de variables categóricas consistente, homogéneo y preparado para la imputación de valores faltantes, el análisis exploratorio de datos (EDA) y el entrenamiento de modelos de Machine Learning.

In [ ]:
# ==========================================================
# LISTADO DE CATEGORÍAS POR VARIABLE CATEGÓRICA
# DataFrame: bd
# ==========================================================

import pandas as pd

print("="*80)
print("LISTADO DE CATEGORÍAS DE LAS VARIABLES CATEGÓRICAS")
print("="*80)

# ==========================================================
# PASO 1
# Seleccionar variables categóricas
# ==========================================================

variables_categoricas = bd.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

# Excluir variable identificadora
if "id_registro" in variables_categoricas:
    variables_categoricas.remove("id_registro")

# ==========================================================
# PASO 2
# Mostrar categorías de cada variable
# ==========================================================

for variable in variables_categoricas:

    categorias = (
        bd[variable]
        .drop_duplicates()
        .sort_values(
            na_position="last",
            key=lambda x: x.astype("string")
        )
        .tolist()
    )

    categorias = [
        "NaN" if pd.isna(valor) else str(valor)
        for valor in categorias
    ]

    print("\n" + "="*80)
    print(f"VARIABLE: {variable}")
    print("="*80)

    for i, categoria in enumerate(categorias, start=1):
        print(f"{i}. {categoria}")

# ==========================================================
# PASO 3
# Resumen
# ==========================================================

print("\n" + "="*80)
print("RESUMEN")
print("="*80)

print(f"Variables categóricas analizadas: {len(variables_categoricas)}")

print("\nProceso finalizado correctamente.")

LISTADO DE CATEGORÍAS DE LAS VARIABLES CATEGÓRICAS

VARIABLE: tipo_inmueble
1.   
2.  - 
3.  Apartamento 
4.  Casa Unifamiliar 
5.  N/A 
6.  NO DATA 
7.  PequeÃ±o Establecimiento Comercial 
8.  Pequeño Establecimiento Comercial 
9.  s/d 
10.  sin dato 
11. -
12. APARTAMENTO
13. Apartamento
14. CASA UNIFAMILIAR
15. Casa Unifamiliar
16. NO DATA
17. PEQUEÑO ESTABLECIMIENTO COMERCIAL
18. PequeÃ±o Establecimiento Comercial
19. Pequeño Establecimiento Comercial
20. S/D
21. SIN DATO
22. apartamento
23. casa unifamiliar
24. no data
25. pequeño establecimiento comercial
26. s/d
27. sin dato
28. NaN

VARIABLE: zona
1.   
2.  - 
3.  N/A 
4.  NO DATA 
5.  Suburbana 
6.  Urbana Costera 
7.  Urbana Interior 
8.  s/d 
9.  sin dato 
10. -
11. NO DATA
12. S/D
13. SIN DATO
14. SUBURBANA
15. Suburbana
16. URBANA COSTERA
17. URBANA INTERIOR
18. Urbana Costera
19. Urbana Interior
20. no data
21. s/d
22. sin dato
23. suburbana
24. urbana costera
25. urbana interior
26. NaN

VARIABLE: nivel_socioeconomico
1.

In [ ]:
# ==========================================================
# ESTANDARIZACIÓN DE VARIABLES CATEGÓRICAS
# DataFrame: bd
# ==========================================================

print("="*80)
print("ESTANDARIZACIÓN DE VARIABLES CATEGÓRICAS")
print("="*80)

# ==========================================================
# DICCIONARIOS DE CATEGORÍAS VÁLIDAS
# ==========================================================

diccionarios = {

    "tipo_inmueble": {
        "apartamento": "Apartamento",
        "casa unifamiliar": "Casa Unifamiliar",
        "pequeño establecimiento comercial": "Pequeño Establecimiento Comercial",
        "pequeã±o establecimiento comercial": "Pequeño Establecimiento Comercial"
    },

    "zona": {
        "suburbana": "Suburbana",
        "urbana costera": "Urbana Costera",
        "urbana interior": "Urbana Interior"
    },

    "nivel_socioeconomico": {
        "alto": "Alto",
        "medio": "Medio",
        "bajo": "Bajo"
    },

    "mes_referencia": {

        "01":"Enero","ene":"Enero","enero":"Enero","january":"Enero",

        "02":"Febrero","feb":"Febrero","febrero":"Febrero","february":"Febrero",

        "03":"Marzo","mar":"Marzo","marzo":"Marzo","march":"Marzo",

        "04":"Abril","abr":"Abril","abril":"Abril","april":"Abril",

        "05":"Mayo","mayo":"Mayo","may":"Mayo",

        "06":"Junio","jun":"Junio","junio":"Junio","june":"Junio",

        "07":"Julio","jul":"Julio","julio":"Julio","july":"Julio",

        "08":"Agosto","ago":"Agosto","agosto":"Agosto","august":"Agosto",

        "09":"Septiembre","sep":"Septiembre","septiembre":"Septiembre","september":"Septiembre",

        "10":"Octubre","oct":"Octubre","octubre":"Octubre","october":"Octubre",

        "11":"Noviembre","nov":"Noviembre","noviembre":"Noviembre","november":"Noviembre",

        "12":"Diciembre","dic":"Diciembre","diciembre":"Diciembre","december":"Diciembre"
    },

    "horario_pico_uso": {
        "madrugada":"Madrugada",
        "mañana":"Mañana",
        "tarde":"Tarde",
        "noche":"Noche"
    },

    "aislamiento_termico": {
        "bueno":"Bueno",
        "regular":"Regular",
        "malo":"Malo"
    },

    "fuente_energia_secundaria": {
    "ninguna":"Ninguna",
    "panel solar":"Panel Solar",
    "planta eléctrica":"Planta Eléctrica",
    "planta electrica":"Planta Eléctrica",
    "planta elã©ctrica":"Planta Eléctrica",
    "inversor con baterías":"Inversor con baterías",
    "inversor con baterias":"Inversor con baterías"
},

    "perfil_energetico": {
        "eficiente":"Eficiente",
        "moderado":"Moderado",
        "ineficiente":"Ineficiente"
    }

}

# ==========================================================
# ESTANDARIZACIÓN
# ==========================================================

reporte = []

for variable, diccionario in diccionarios.items():

    if variable not in bd.columns:
        continue

    originales = bd[variable].copy()

    serie = (
        bd[variable]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    estandarizada = serie.map(diccionario)

    corregidos = (
        originales.notna() &
        estandarizada.notna() &
        (originales.astype("string") != estandarizada.astype("string"))
    ).sum()

    invalidados = (
        originales.notna() &
        estandarizada.isna()
    ).sum()

    bd[variable] = estandarizada.astype("string")

    reporte.append({

        "Variable": variable,

        "Valores corregidos": int(corregidos),

        "Convertidos a NaN": int(invalidados),

        "Categorías finales": bd[variable].dropna().nunique()

    })

# ==========================================================
# REPORTE
# ==========================================================

print("\nResumen de la estandarización\n")

display(pd.DataFrame(reporte))

print("\nProceso finalizado correctamente.")

ESTANDARIZACIÓN DE VARIABLES CATEGÓRICAS

Resumen de la estandarización



,Variable,Valores corregidos,Convertidos a NaN,Categorías finales
0,tipo_inmueble,9906,473,3
1,zona,6599,1447,3
2,nivel_socioeconomico,6538,1992,3
3,mes_referencia,9610,494,12
4,horario_pico_uso,6615,1474,4
5,aislamiento_termico,6112,4838,3
6,fuente_energia_secundaria,23991,1472,4
7,perfil_energetico,6626,1204,3



Proceso finalizado correctamente.


## 5. Detección y estandarización de valores nulos

## Objetivo

Este script tiene como finalidad identificar y estandarizar los valores utilizados para representar datos faltantes dentro del conjunto de datos. Para ello, detecta valores conocidos como **nulos fantasma** (por ejemplo `"N/A"`, `"NULL"` o `-999`) y los reemplaza por el valor estándar `NaN` de NumPy, permitiendo que sean reconocidos correctamente por Pandas durante las etapas posteriores del análisis.

A diferencia del script de estandarización de variables categóricas, este procedimiento únicamente trata representaciones generales de valores faltantes y no modifica categorías válidas.

---

# Flujo de funcionamiento

## Paso 1. Definición de los valores fantasma

Se construye una lista con todos los valores que serán considerados como representaciones de datos faltantes.

Entre ellos se incluyen:

- Cadenas vacías.
- Signos de interrogación utilizados como marcadores de ausencia.
- Valores de texto como `"N/A"`, `"NULL"` o `"None"`.
- Valores numéricos empleados frecuentemente como códigos de ausencia (`-999` y `9999`).

Esta lista constituye el criterio utilizado durante la búsqueda de valores faltantes.

---

## Paso 2. Búsqueda de valores fantasma

El script recorre todas las columnas del conjunto de datos y contabiliza cuántas veces aparece cada uno de los valores definidos anteriormente.

Los resultados se almacenan en una tabla que muestra:

- Variable.
- Valor detectado.
- Cantidad de ocurrencias.

Este reporte permite conocer la distribución de los valores faltantes antes de su estandarización.

---

## Paso 3. Conversión a valores nulos

Una vez identificados los valores fantasma, el script reemplaza todas sus apariciones por el valor estándar `NaN`.

Este procedimiento unifica todas las representaciones de datos faltantes bajo un único formato compatible con las funciones de análisis de Pandas.

---

## Paso 4. Resumen del proceso

Después de realizar la conversión, el script calcula:

- Número total de valores nulos antes del proceso.
- Número total de valores nulos después del proceso.
- Cantidad de nuevos valores `NaN` generados.

Este resumen permite evaluar el impacto de la estandarización.

---

## Paso 5. Reporte final de valores faltantes

Finalmente se genera una tabla resumen para cada variable del conjunto de datos indicando:

- Nombre de la variable.
- Cantidad de valores nulos.
- Porcentaje de valores nulos respecto al total de registros.

La tabla se ordena de forma descendente según el porcentaje de valores faltantes, facilitando la identificación de las variables que requerirán mayor atención durante la etapa de imputación.

---

# Resultado

Al finalizar la ejecución del script:

- Todas las representaciones generales de datos faltantes quedan convertidas al valor estándar `NaN`.
- Se obtiene un reporte detallado de los valores detectados y reemplazados.
- Se genera un resumen del estado actual de los valores faltantes para cada variable.
- El conjunto de datos queda preparado para las etapas posteriores de imputación y análisis exploratorio de datos (EDA).

In [ ]:
# ==========================================================
# PASO 2
# Detección y estandarización de valores nulos
# ==========================================================

print("\n[ PASO 2 ] Detección y estandarización de valores nulos")

# ==========================================================
# Valores considerados como nulos fantasma
# ==========================================================

valores_fantasma = [
    "",
    "?",
    "??",
    "N/A",
    "n/a",
    "NA",
    "na",
    "NULL",
    "null",
    "None",
    "none",
    "N/D",
    "-999",
    -999,
    "9999",
    9999
]

# ----------------------------------------------------------
# REPORTE DE NULOS FANTASMA ENCONTRADOS
# ----------------------------------------------------------

print("\nBuscando valores considerados como nulos...")

reporte_fantasma = []

for columna in bd.columns:

    for valor in valores_fantasma:

        cantidad = (bd[columna] == valor).sum()

        if cantidad > 0:

            reporte_fantasma.append({

                "Variable": columna,

                "Valor detectado": str(valor),

                "Cantidad": int(cantidad)

            })

reporte_fantasma = pd.DataFrame(reporte_fantasma)

if len(reporte_fantasma) > 0:

    print("\nValores fantasma detectados:")

    display(

        reporte_fantasma.sort_values(

            by=["Variable","Cantidad"],

            ascending=[True,False]

        )

    )

else:

    print("\nNo se detectaron valores fantasma.")

# ----------------------------------------------------------
# Conversión a NaN
# ----------------------------------------------------------

nulos_antes = bd.isna().sum().sum()

bd.replace(valores_fantasma, np.nan, inplace=True)

nulos_despues = bd.isna().sum().sum()

print("\nResumen de la estandarización")

print(f"Nulos originales            : {nulos_antes}")
print(f"Nulos después del proceso   : {nulos_despues}")
print(f"Nuevos NaN generados        : {nulos_despues-nulos_antes}")

# ----------------------------------------------------------
# REPORTE FINAL
# ----------------------------------------------------------

reporte_nulos = pd.DataFrame({

    "Variable": bd.columns,

    "Cantidad de nulos": bd.isna().sum().values,

    "Porcentaje (%)": (

        bd.isna()

          .mean()

          .mul(100)

          .round(2)

          .values

    )

})

print("\nEstado de los valores faltantes:")

display(

    reporte_nulos.sort_values(

        by="Porcentaje (%)",

        ascending=False

    )

)


[ PASO 2 ] Detección y estandarización de valores nulos

Buscando valores considerados como nulos...

No se detectaron valores fantasma.

Resumen de la estandarización
Nulos originales            : 172855
Nulos después del proceso   : 172855
Nuevos NaN generados        : 0

Estado de los valores faltantes:


,Variable,Cantidad de nulos,Porcentaje (%)
34,certificacion_energetica_previa,14284,14.73
9,temperatura_promedio_c,10347,10.67
28,aislamiento_termico,9696,10.00
29,antiguedad_electrodomesticos_anios,8728,9.00
5,superficie_m2,6982,7.20
40,consumo_kwh_por_m2,6714,6.92
10,tiene_aire_acondicionado,6642,6.85
39,costo_estimado_usd,6503,6.71
37,consumo_kwh_mensual,6500,6.70
14,tiene_lavadora,6424,6.63


## 6. Validación de rangos y coherencia física (Outliers por dominio)

Una vez estandarizados los valores nulos y normalizados los tipos de datos de acuerdo con el diccionario oficial, se ejecutó una etapa de validación lógica para garantizar la viabilidad física y coherencia de los valores registrados en las variables numéricas. En conjuntos de datos energéticos, es de vital importancia identificar y eliminar registros que presenten magnitudes físicamente imposibles (por ejemplo, porcentajes superiores al 100 %, horas de uso diario mayores a 24 horas o días de corte superiores a los días de un mes), los cuales suelen originarse por errores de transcripción, fallos de sensores o artefactos durante la recolección de datos.

El procedimiento se desarrolló bajo el siguiente protocolo:

1. **Definición de fronteras teóricas por dominio:** Se estableció un diccionario de rangos admisibles `[mínimo, máximo]` para cada variable continua y discreta del conjunto de datos, fundamentado en las leyes físicas del problema y el contexto geográfico (clima tropical en República Dominicana). Por ejemplo:
   - **Porcentajes:** `pct_iluminacion_led` restringido estrictamente al intervalo `[0.0, 100.0]`.
   - **Límites temporales diarios:** Variables de uso diario como `horas_uso_aa_dia`, `horas_uso_iluminacion_dia` y los perfiles por ambiente limitadas al intervalo `[0.0, 24.0]` horas.
   - **Límites temporales mensuales:** `dias_sin_electricidad_mes` acotado a un máximo de `31` días, y `horas_uso_planta_o_inversor_mes` a un máximo de `744` horas ($31 \text{ días} \times 24 \text{ horas}$).
   - **Magnitudes físicas:** Consumos, superficies y conteos de equipos restringidos a valores mayores o iguales a cero (`≥ 0`).

2. **Detección y depuración de valores fuera de rango:** Se evaluó cada variable contra sus fronteras teóricas. Todos los valores identificados fuera de los límites permisibles fueron contabilizados e invalidados, reemplazándose estrictamente por el valor nulo **`np.nan`** de NumPy. Esta decisión metodológica unifica la representación de ausencias en toda la base de datos, conserva la información válida del resto de las columnas del registro y traslada el tratamiento del dato atípico a la fase posterior de imputación.

Asociado a esta depuración, se generó un reporte de auditoría que detalla la cantidad exacta de valores depurados por atributo, garantizando que el conjunto de datos resultante cumpla rigurosamente con las restricciones del dominio del problema antes de proceder al Análisis Exploratorio de Datos (EDA).

In [ ]:
# ==========================================================
# VALIDACIÓN Y DEPURACIÓN DE RANGOS LÓGICOS POR DOMINIO
# DataFrame: bd
# ==========================================================

print("="*80)
print("DEPURACIÓN DE VALORES FUERA DE RANGO (OUTLIERS POR DOMINIO)")
print("="*80)

# ==========================================================
# 1. Definición de rangos válidos [mínimo, máximo]
# ==========================================================
rangos_validos = {
    # Características del inmueble y contexto
    "num_personas": (1, 100),                      # Mínimo 1 ocupante/empleado
    "superficie_m2": (10.0, 10000.0),              # Área útil lógica en m2
    "antiguedad_construccion_anios": (0, 150),     # Años de construcción
    "dias_facturacion": (15, 45),                  # Ciclo de facturación (típicamente 28-31)
    "temperatura_promedio_c": (15.0, 45.0),        # Clima tropical (República Dominicana)

    # Equipos e iluminación
    "cantidad_unidades_aa": (0, 50),               # Unidades de aire acondicionado
    "horas_uso_aa_dia": (0.0, 24.0),               # Máximo 24 horas al día
    "cantidad_tv_o_pantallas": (0, 50),
    "cantidad_computadoras": (0, 50),
    "pct_iluminacion_led": (0.0, 100.0),           # Porcentaje estricto de 0 a 100%
    "cantidad_focos": (0, 500),
    "horas_uso_iluminacion_dia": (0.0, 24.0),      # Máximo 24 horas al día
    "otros_equipos_pequenos": (0, 200),
    "cantidad_equipos_total": (0, 1000),
    "antiguedad_electrodomesticos_anios": (0.0, 50.0),

    # Perfil de horas por ambiente (Máximo 24h diarias por zona)
    "horas_dia_cocina": (0.0, 24.0),
    "horas_dia_sala_estar": (0.0, 24.0),
    "horas_dia_dormitorios": (0.0, 24.0),
    "horas_dia_oficina_estudio": (0.0, 24.0),
    "horas_dia_lavanderia": (0.0, 24.0),

    # Cortes y respaldo energético (Mensual)
    "dias_sin_electricidad_mes": (0, 31),          # Máximo 31 días al mes
    "horas_uso_planta_o_inversor_mes": (0.0, 744.0), # Máximo 31 días * 24 horas = 744h

    # Consumos y costos energéticos
    "generacion_solar_kwh_mensual": (0.0, 20000.0),
    "consumo_kwh_mes_anterior": (0.0, 50000.0),
    "variacion_pct_consumo_mensual": (-100.0, 1000.0), # Variación porcentual
    "consumo_kwh_mensual": (0.0, 50000.0),         # Consumo central no negativo
    "consumo_neto_facturado_kwh": (-5000.0, 50000.0),# Puede ser negativo en inyección solar
    "costo_estimado_usd": (0.0, 50000.0),          # Costo no negativo
    "consumo_kwh_por_m2": (0.0, 2000.0),
    "consumo_kwh_por_persona": (0.0, 10000.0)
}

# ==========================================================
# 2. Evaluación y filtrado por columna
# ==========================================================
reporte_rangos = []
total_invalidados = 0

for columna, (val_min, val_max) in rangos_validos.items():

    if columna in bd.columns:
        # Identificar registros no nulos que escapan del rango permitido
        serie_temporal = bd[columna].dropna()
        mask_fuera_rango = (bd[columna] < val_min) | (bd[columna] > val_max)
        n_fuera_rango = mask_fuera_rango.sum()

        if n_fuera_rango > 0:
            # Capturar ejemplos de valores inválidos para la auditoría
            ejemplos = bd.loc[mask_fuera_rango, columna].dropna().unique()[:3]
            ejemplos_str = ", ".join([str(val) for val in ejemplos])

            # Reemplazar valores fuera de rango estrictamente por np.nan
            bd.loc[mask_fuera_rango, columna] = np.nan

            total_invalidados += n_fuera_rango
        else:
            ejemplos_str = "Ninguno"

        reporte_rangos.append({
            "Variable": columna,
            "Rango Admisible": f"[{val_min}, {val_max}]",
            "Valores Eliminados": n_fuera_rango,
            "Ejemplos Invalidados": ejemplos_str
        })

# ==========================================================
# 3. Resumen y visualización de resultados
# ==========================================================
df_reporte = pd.DataFrame(reporte_rangos)

print(f"\n-> Proceso completado. Total de valores fuera de rango eliminados (convertidos a np.nan): {total_invalidados:,}\n")
print("Reporte detallado por variable:")

display(
    df_reporte.sort_values(by="Valores Eliminados", ascending=False)
)

# Verificación de integridad estructural del DataFrame
print(f"\nDimensiones finales del dataset preservadas: {bd.shape[0]:,} filas x {bd.shape[1]} columnas.")

DEPURACIÓN DE VALORES FUERA DE RANGO (OUTLIERS POR DOMINIO)

-> Proceso completado. Total de valores fuera de rango eliminados (convertidos a np.nan): 2,636

Reporte detallado por variable:


,Variable,Rango Admisible,Valores Eliminados,Ejemplos Invalidados
1,superficie_m2,"[10.0, 10000.0]",517,"3.4, 2.0, 1.3"
6,horas_uso_aa_dia,"[0.0, 24.0]",468,"85.3, 85.8, 26.6"
9,pct_iluminacion_led,"[0.0, 100.0]",388,"174.7, 178.6, 130.3"
27,costo_estimado_usd,"[0.0, 50000.0]",369,"-218.25, -594.9, -407.78"
2,antiguedad_construccion_anios,"[0, 150]",312,"-1.0, 500.0, 220.0"
20,dias_sin_electricidad_mes,"[0, 31]",289,"60.0, 35.0, 45.0"
0,num_personas,"[1, 100]",229,"0.0, 150.0, 300.0"
13,cantidad_equipos_total,"[0, 1000]",37,"-3.0, -1.0, -2.0"
25,consumo_kwh_mensual,"[0.0, 50000.0]",27,"88133.3, 67334.0, 52519.1"
5,cantidad_unidades_aa,"[0, 50]",0,Ninguno



Dimensiones finales del dataset preservadas: 96,961 filas x 43 columnas.


## 7. Depuración de estricta binaridad (Estandarización de variables booleanas)

Una vez finalizada la etapa de validación relacional, se ejecutó un proceso de **depuración de estricta binaridad** sobre todas las variables categóricas de naturaleza booleana. El objetivo de esta fase es garantizar que dichas variables conserven únicamente los valores permitidos dentro del dominio del problema, evitando que errores de captura, importación o transformación introduzcan categorías inválidas que puedan afectar el análisis estadístico o el entrenamiento de modelos de aprendizaje automático.

A diferencia de las validaciones relacionales, este procedimiento evalúa de forma **individual** cada variable binaria, verificando que sus observaciones pertenezcan exclusivamente al conjunto de valores admitidos.

El flujo general del proceso es el siguiente:

1. **Identificación de valores inválidos:** Se inspecciona cada variable binaria para localizar registros cuyo valor no corresponda a una representación válida del estado lógico esperado.

2. **Validación del dominio binario:** Se consideran válidos únicamente los valores equivalentes a `0` y `1`, incluyendo sus representaciones compatibles (`0.0`, `1.0`, `False` y `True`) para absorber diferencias derivadas de conversiones de tipo de dato.

3. **Neutralización de inconsistencias:** Todo valor distinto al dominio binario permitido es reemplazado por `np.nan`, preservando el registro completo para que la información faltante pueda ser tratada posteriormente mediante técnicas de imputación.

4. **Auditoría del proceso:** Finalmente, se genera un reporte por variable indicando la cantidad de valores corregidos, ejemplos de los valores reemplazados y el número total de valores nulos resultantes. Como verificación final, se inspeccionan los valores únicos restantes para confirmar que únicamente permanezcan `0`, `1` y valores nulos.


### Variables Auditadas

La validación de binaridad se aplica sobre las siguientes variables del conjunto de datos:

- `tiene_aire_acondicionado`
- `tiene_calentador_agua_electrico`
- `tiene_lavadora`
- `certificacion_energetica_previa`


### Regla de Integridad Aplicada

Para cada una de las variables binarias, se verifica el cumplimiento de la siguiente condición:

$$
x \in \{0,\;1\}
$$

considerando también como representaciones válidas:

$$
\{0,\;1,\;0.0,\;1.0,\;\text{False},\;\text{True}\}
$$

Cualquier observación que no pertenezca a este conjunto es considerada una inconsistencia de dominio y se invalida mediante la asignación de `np.nan`.

Esta estrategia garantiza que todas las variables binarias mantengan un dominio homogéneo antes de continuar con las etapas de imputación, codificación y entrenamiento de modelos predictivos.

In [ ]:
# ==========================================================
# DEPURACIÓN DE ESTRICTA BINARIDAD (VALORES 0 Y 1)
# DataFrame: bd
# ==========================================================

print("="*80)
print("DEPURACIÓN DE VARIABLES BINARIAS (ESTANDARIZACIÓN A 0 Y 1)")
print("="*80)

# Arreglo de variables binarias del proyecto
variables_binarias_totales = [
    "tiene_aire_acondicionado",
    "tiene_calentador_agua_electrico",
    "tiene_lavadora",
    "certificacion_energetica_previa"
]

total_depurados_bin = 0
reporte_binarias = []

for col in variables_binarias_totales:
    if col in bd.columns:
        # 1. Identificar valores que NO son nulos y que NO son ni 0 ni 1
        # Se incluyen variaciones flotantes (0.0, 1.0) y lógicas (True, False) por seguridad
        mask_validos = bd[col].isin([0, 1, 0.0, 1.0, True, False])
        mask_invalido = bd[col].notna() & (~mask_validos)

        n_invalido = mask_invalido.sum()

        if n_invalido > 0:
            # Capturar hasta 3 ejemplos de valores erróneos antes de borrarlos
            ejemplos = bd.loc[mask_invalido, col].unique()[:3]
            ejemplos_str = ", ".join([str(v) for v in ejemplos])

            # 2. Reemplazar los valores inválidos estrictamente por np.nan
            bd.loc[mask_invalido, col] = np.nan
            total_depurados_bin += n_invalido
        else:
            ejemplos_str = "Ninguno"

        reporte_binarias.append({
            "Variable Binaria": col,
            "Valores Inválidos Eliminados": n_invalido,
            "Ejemplos Reemplazados por NaN": ejemplos_str,
            "Nulos Totales Resultantes": bd[col].isna().sum()
        })

# 3. Presentación de resultados
df_reporte_bin = pd.DataFrame(reporte_binarias)

print(f"\n-> Depuración completada. Total de valores no binarios convertidos a np.nan: {total_depurados_bin:,}\n")
print("Reporte de auditoría por columna binaria:")
display(df_reporte_bin)

# Verificación de valores únicos finales para garantizar que solo queden 0, 1 y nulos
print("\nVerificación de valores únicos restantes por columna:")
for col in variables_binarias_totales:
    if col in bd.columns:
        val_unicos = bd[col].dropna().unique()
        print(f"   - [{col}]: {sorted(list(val_unicos))}")
print("="*80 + "\n")

DEPURACIÓN DE VARIABLES BINARIAS (ESTANDARIZACIÓN A 0 Y 1)

-> Depuración completada. Total de valores no binarios convertidos a np.nan: 0

Reporte de auditoría por columna binaria:


,Variable Binaria,Valores Inválidos Eliminados,Ejemplos Reemplazados por NaN,Nulos Totales Resultantes
0,tiene_aire_acondicionado,0,Ninguno,6642
1,tiene_calentador_agua_electrico,0,Ninguno,6390
2,tiene_lavadora,0,Ninguno,6424
3,certificacion_energetica_previa,0,Ninguno,14284



Verificación de valores únicos restantes por columna:
   - [tiene_aire_acondicionado]: [np.float64(0.0), np.float64(1.0)]
   - [tiene_calentador_agua_electrico]: [np.float64(0.0), np.float64(1.0)]
   - [tiene_lavadora]: [np.float64(0.0), np.float64(1.0)]
   - [certificacion_energetica_previa]: [np.float64(0.0), np.float64(1.0)]



In [ ]:
total_original = len(bd)

print("LIMPIEZA ESTRUCTURAL: ELIMINACIÓN DE FILAS COMPLETAMENTE VACÍAS Y ELIMINACIÓN DE FILAS CON VARIABLE PREDICTIVA VACÍA")

# Crea arreglo de columnas auditadas
cols_variables_reales = [col for col in bd.columns if col != 'id_registro']

bd = bd.dropna(subset=cols_variables_reales, how='all').reset_index(drop=True)
bd = bd.dropna(subset=["perfil_energetico"]).reset_index(drop=True)

filas_borradas = total_original - len(bd)
print(f"Total de registros originales : {total_original:,}")
print(f"Filas 100% vacías eliminadas  : {filas_borradas:,}")
print(f"Total de registros limpios    : {len(bd):,}\n")

LIMPIEZA ESTRUCTURAL: ELIMINACIÓN DE FILAS COMPLETAMENTE VACÍAS Y ELIMINACIÓN DE FILAS CON VARIABLE PREDICTIVA VACÍA
Total de registros originales : 96,961
Filas 100% vacías eliminadas  : 2,424
Total de registros limpios    : 94,537



## 8. Exportación de la base de datos

In [ ]:
# ==========================================================
# EXPORTAR DATASET LIMPIO (EN LA CARPETA DEL NOTEBOOK)
# ==========================================================

nombre_archivo = "dataset_consumo_energetico_LIMPIO.csv"

# 1. Guardar el DataFrame localmente (al lado de este notebook)
bd.to_csv(
    nombre_archivo,
    index=False,
    encoding="utf-8-sig"
)

# 2. Obtener la ruta exacta de tu disco duro donde quedó guardado
ruta_completa = os.path.abspath(nombre_archivo)

print("="*80)
print("DATASET EXPORTADO CORRECTAMENTE")
print("="*80)
print(f"Archivo generado : {nombre_archivo}")
print(f"Ubicación en PC  : {ruta_completa}")
print(f"Registros        : {bd.shape[0]}")
print(f"Variables        : {bd.shape[1]}")

DATASET EXPORTADO CORRECTAMENTE
Archivo generado : dataset_consumo_energetico_LIMPIO.csv
Ubicación en PC  : /content/dataset_consumo_energetico_LIMPIO.csv
Registros        : 94537
Variables        : 43
